In [1]:
# ===========================================
# Libraries
# ===========================================

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid, cumulative_trapezoid
from scipy.special import erf
import emcee
import time
from getdist import MCSamples, plots
import matplotlib.pyplot as plt

In [2]:
# ==========================================
# Loading CC data
# ==========================================

cc_dat_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/CC/CC_Hz_data.txt"
df_cc = pd.read_csv(cc_dat_path, sep=r"\s+")

z_cc   = df_cc["z_cc"].to_numpy(float)
H_obs  = df_cc["H_cc"].to_numpy(float)
H_err   = df_cc["H_cc_err"].to_numpy(float)

# ====================================================
# CCsubset correction: remove correlated CC measurements
# ====================================================
# Moresco (arXiv:2412.01994), Table 1, footnote (a): Borghi et al. 2022
# (Lick indices, z=0.75), Jimenez et al. 2023 (Machine Learning, z=0.75),
# and Jiao et al. 2023 (Full Spectral Fitting, z=0.80) are all derived
# from, or calibrated on, the same LEGA-C survey sample -- Jimenez et al.
# train their ML method directly on the LEGA-C-based ages of Borghi et al.,
# and Jiao et al. also apply their method to the LEGA-C sample. Moresco
# explicitly recommends against using such same-sample/different-method
# points jointly as independent measurements.
#
# We keep Borghi et al. 2022 (z=0.75, H=98.8) -- the direct spectroscopic
# (Lick-index) measurement -- and drop Jimenez et al. 2023 (z=0.75,
# H=105.0, an ML re-derivation calibrated on the same underlying ages) and
# Jiao et al. 2023 (z=0.80, H=113.1, same LEGA-C sample), reducing the CC
# sample from 35 to 33 points.

drop_mask  = np.isclose(z_cc, 0.75, atol=1e-4) & np.isclose(H_obs, 105.0, atol=1e-4)   # Jimenez_2023
drop_mask |= np.isclose(z_cc, 0.80, atol=1e-4) & np.isclose(H_obs, 113.1, atol=1e-4)   # Jiao_2023

print(f"CC points before CCsubset correction: {len(z_cc)}")
z_cc, H_obs, H_err = z_cc[~drop_mask], H_obs[~drop_mask], H_err[~drop_mask]
print(f"CC points after CCsubset correction (Jimenez_2023, Jiao_2023 removed): {len(z_cc)}")


# ========================================================
# Loading SNe Ia (Pantheon+SH0ES) data
# ========================================================

sne_dat_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/SNe/Pantheon+SH0ES.dat"
sne_cov_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/SNe/Pantheon+SH0ES_STAT+SYS.cov"
df_sne = pd.read_csv(sne_dat_path, sep=r"\s+", comment="#", engine="python")
raw_cov = np.loadtxt(sne_cov_path, skiprows=1)
Cov_sne_full = raw_cov.reshape((1701, 1701))

mask = df_sne["zCMB"].values > 0.01
df_cut = df_sne[mask].reset_index(drop=True)
Cov_sne = Cov_sne_full[mask, :][:, mask]
z_sne = df_cut["zCMB"].values
mB_obs = df_cut["m_b_corr"].values
Cinv_sne = np.linalg.inv(Cov_sne)


# ========================================================
# Loading DESI BAO DR2 data
# ========================================================

bao_dat_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/BAO/desi_gaussian_bao_ALL_GCcomb_mean.txt"
bao_cov_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/BAO/desi_gaussian_bao_ALL_GCcomb_cov.txt"
df_bao = pd.read_csv(bao_dat_path, sep=r"\s+", comment="#", names=["z", "value", "quantity"])

z_bao = df_bao["z"].values
d_obs = df_bao["value"].values
q_bao = df_bao["quantity"].str.lower().values
cov_bao = np.loadtxt(bao_cov_path)
Cinv_bao = np.linalg.inv(cov_bao)


# ====================================================
# Loading FRB data
# ====================================================

frb_dat_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/FRB/frb_catalog.txt"
df_frb       = pd.read_csv(frb_dat_path, sep=r"\t", engine="python")
df_frb       = df_frb.sort_values(by="Redshift", ascending=True).reset_index(drop=True)

z_frb        = df_frb["Redshift"].to_numpy(float)
DM_obs       = df_frb["DM_obs"].to_numpy(float)
DM_MW        = df_frb["DM_MW_ISM_(NE2001)"].to_numpy(float)
# DMbudget robustness test (Likelihood-3-style, Sec. 3.3 of arXiv:2410.01974):
# DM_MW,halo is no longer subtracted here as a fixed 50 pc/cm^3 constant --
# it is instead marginalized inside the likelihood via P_halo(DM_halo),
# jointly with DM_host, rather than being fixed a priori (see Fiducial/Priors
# and likelihood cells below).
DM_ext_obs   = DM_obs - DM_MW        # DM_cos + DM_halo + DM_host/(1+z) [pc/cm^3]

CC points before CCsubset correction: 35
CC points after CCsubset correction (Jimenez_2023, Jiao_2023 removed): 33


In [3]:
# ====================================================
# Fiducial / Priors
# ====================================================

# Uniform priors
H0_min,       H0_max       = 50.0,  90.0
Om0_min,      Om0_max      = 0.1,   0.6
Ob_min,       Ob_max       = 0.01,  0.1
MB_min,       MB_max       = -20.0, -18.0
sig_host_min, sig_host_max = 0.2,   2.0
e_mu_min,     e_mu_max     = 20.0,  200.0

# Physical constants
c_kms = 299792.458         # speed of light [km/s]
c_si  = 2.99792458e8       # speed of light [m/s]
G_si  = 6.67430e-11        # gravitational constant [m^3 kg^-1 s^-2]
mp_si = 1.67262192369e-27  # proton mass [kg]

# Unit conversions
km_to_pc  = 1.0 / (3.0857e13)
m2_to_cm2 = 1e4
Mpc_to_cm = 3.0857e24

# Astrophysical/cosmological parameters
f_d = 0.83
chi_e = 0.875
Neff  = 3.046

# ====================================================
# Robustness test (DMbudget): Galactic DM budget uncertainties
# ====================================================
# sigma_MW: NE2001 DM_MW,ISM systematic uncertainty [pc/cm^3], following
#           Manchester et al. (2005, ATNF Pulsar Catalogue) calibration
#           adopted in arXiv:2410.01974. Enters in quadrature with the
#           statistical sigma_cos term (as in all three Likelihoods of
#           arXiv:2410.01974 -- DM_MW,ISM is never marginalized by
#           convolution there, only DM_Host and, in Likelihood 3, DM_Halo).
#
# DM_MW,halo: marginalized via a Gaussian P_halo(DM_halo), following the
#             "Likelihood 3" treatment of Sec. 3.3 of arXiv:2410.01974
#             (full convolution instead of a quadrature approximation).
#             mu_halo = 50 pc/cm^3 matches our own adopted fiducial value;
#             sigma_halo is the effective std of our own quoted plausible
#             range DM_MW,halo ~ U[25,110] pc/cm^3 (Sec. 5 of the paper),
#             std = (max-min)/sqrt(12) -- same range, now propagated by
#             convolution rather than by a quadrature term.
#
# npts_halo=30: the double convolution (DM_halo x DM_host, see likelihood
# cell) is ~30x more expensive per likelihood call than the single-host
# convolution used before, so the grid is kept modest here -- still >120
# points/sigma_halo of resolution across a +/-5sigma window, more than
# enough given how smooth P_halo and P_cos are.

sigma_MW = 30.0

mu_halo = 50.0
DM_halo_min, DM_halo_max = 25.0, 110.0
sigma_halo = (DM_halo_max - DM_halo_min) / np.sqrt(12.0)

npts_halo    = 30
DM_halo_grid = np.linspace(max(0.0, mu_halo - 5.0 * sigma_halo),
                            mu_halo + 5.0 * sigma_halo, npts_halo)
P_halo_grid  = (1.0 / (np.sqrt(2 * np.pi) * sigma_halo)) \
               * np.exp(-0.5 * ((DM_halo_grid - mu_halo) / sigma_halo)**2)

print(f"sigma_MW     = {sigma_MW:.3f} pc/cm^3")
print(f"mu_halo      = {mu_halo:.3f} pc/cm^3")
print(f"sigma_halo   = {sigma_halo:.3f} pc/cm^3  (from U[{DM_halo_min:.0f},{DM_halo_max:.0f}])")
print(f"DM_halo_grid = [{DM_halo_grid[0]:.2f}, {DM_halo_grid[-1]:.2f}] pc/cm^3, {npts_halo} pts")


# ====================================================
# Redshift / scale-factor grids
# ====================================================

zi, zf = 0.0, 2.5
N_grid = 300
z_grid = np.linspace(zi, zf, N_grid)
dz     = np.diff(z_grid)

sigma_MW     = 30.000 pc/cm^3
mu_halo      = 50.000 pc/cm^3
sigma_halo   = 24.537 pc/cm^3  (from U[25,110])
DM_halo_grid = [0.00, 172.69] pc/cm^3, 30 pts


In [4]:
# ========================================================
# Hubble functions
# ========================================================

def E_LCDM(z, Om0):
    Om = Om0 * (1 + z)**3
    Ol0 = 1 - Om0
    return np.sqrt(Om + Ol0)

def H_LCDM(z, H0, Om0):
    return H0 * E_LCDM(z, Om0)

In [5]:
# ========================================================
# Luminosity distance
# ========================================================

def dL_LCDM(z, H0, Om0):
    prefactor  = c_kms * (1.0 + z)
    H_grid     = H_LCDM(z_grid, H0, Om0)
    integrand  = 1.0 / H_grid
    integral   = cumulative_trapezoid(integrand, z_grid, initial=0.0)
    integral_z = np.interp(z, z_grid, integral)
    return prefactor * integral_z
    

# ========================================================
# Apparent magnitude
# ========================================================

def mB_LCDM(z, H0, Om0, MB):
    dL = dL_LCDM(z, H0, Om0)
    mu = 5.0 * np.log10(dL) + 25.0 
    return mu + MB

In [6]:
# ========================================================
# BAO functions
# ========================================================

# Comoving distance
def comoving_dist(z, H0, Om0):
    E_vals = E_LCDM(z_grid, Om0)
    integrand = 1.0 / E_vals
    integral = np.empty_like(z_grid)
    integral[0] = 0.0
    integral[1:] = np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * dz)
    I_z = np.interp(z, z_grid, integral)
    return (c_kms / H0) * I_z


# Theoretical BAO vector
def d_LCDM(z_arr, q_arr, H0, Om0, Ob):
    h2 = (H0 / 100.0)**2
    Obh2 = Ob * h2
    Omh2 = Om0 * h2
    rd = 147.05 * ((Obh2/0.02236)**-0.13) \
                  * ((Omh2/0.1432)**-0.23) \
                  * ((Neff/3.04)**-0.1)
    dC = comoving_dist(z_arr, H0, Om0)
    H_z = H_LCDM(z_arr, H0, Om0)
    dm = dC / rd
    dh = (c_kms / H_z) / rd
    dv = (z_arr * dm**2 * dh)**(1/3)
    d_bao = np.empty_like(z_arr)
    mask_dv = np.array(["dv" in q for q in q_arr])
    mask_dm = np.array(["dm" in q for q in q_arr])
    mask_dh = np.array(["dh" in q for q in q_arr])
    d_bao[mask_dv] = dv[mask_dv]
    d_bao[mask_dm] = dm[mask_dm]
    d_bao[mask_dh] = dh[mask_dh]
    return d_bao

In [7]:
# ====================================================
# Theoretical cosmological dispersion measure [pc/cm^3]
# ====================================================

def DM_cos_LCDM(z, H0, Om0, Ob):
    prefactor = 3.0 * c_si * f_d * chi_e / (8.0 * np.pi * G_si * mp_si)
    H_grid    = H_LCDM(z_grid, H0, Om0)
    integrand = (1 + z_grid) / H_grid
    I_grid    = cumulative_trapezoid(integrand, z_grid, initial=0.0)
    I_z       = np.interp(z, z_grid, I_grid)
    DM_cos_z    = prefactor * Ob * H0**2 * I_z                      # Before units conversion
    DM_cos_corr = DM_cos_z * (km_to_pc / (m2_to_cm2 * Mpc_to_cm))   # Physical units (pc cm^-3)
    return DM_cos_corr

In [8]:
# ====================================================
# CC + SNe + BAO + FRB likelihood
# ====================================================

# Log-prior
def lnprior_cc_sne_bao_frb(theta):
    H0, Om0, Ob, MB, sigma_host, exp_mu = theta
    if not (H0_min       <= H0         <= H0_max):       return -np.inf
    if not (Om0_min      <= Om0        <= Om0_max):      return -np.inf
    if not (Ob_min       <= Ob         <= Ob_max):       return -np.inf
    if not (MB_min       <= MB         <= MB_max):       return -np.inf
    if not (sig_host_min <= sigma_host <= sig_host_max): return -np.inf
    if not (e_mu_min     <= exp_mu     <= e_mu_max):     return -np.inf
    return 0.0


# Convolved FRB PDF: cosmological (truncated Gaussian) + Galactic halo (Gaussian)
# + host galaxy (log-normal), double-marginalized over DM_halo and DM_host
# (DMbudget robustness test -- "Likelihood 3"-style double convolution,
# Sec. 3.3 of arXiv:2410.01974). DM_halo_grid and P_halo_grid are the fixed
# (theta-independent) grid/weights defined in the Fiducial/Priors cell.
# npts_host=120 (down from the 300 used in the single-convolution fiducial
# model): the extra DM_halo dimension already multiplies the per-call cost
# by npts_halo, so npts_host is kept modest to keep the MCMC runtime
# tractable for this robustness test.
def P_DM_ext_vectorized(z_frb, DM_ext_obs, DM_cos_mean, sigma_cos,
                         sigma_host, exp_mu, npts_host=120):

    mu_host      = np.log(exp_mu)
    upper        = max(DM_ext_obs.max(), 5.0 * exp_mu + DM_halo_grid.max())
    DM_host_grid = np.linspace(1e-2, upper, npts_host)

    # broadcast shapes: (N,1,1) x (1,n_halo,1) x (1,1,n_host) -> (N,n_halo,n_host)
    DM_ext_array       = DM_ext_obs[:, None, None]
    z_array             = z_frb[:, None, None]
    DM_halo_array       = DM_halo_grid[None, :, None]
    DM_host_array       = DM_host_grid[None, None, :]
    sigma_cos_array     = sigma_cos[:, None, None]
    DM_cos_mean_array   = DM_cos_mean[:, None, None]

    DM_cos_array = DM_ext_array - DM_host_array / (1 + z_array) - DM_halo_array

    # standardized variable
    y = (DM_cos_array - DM_cos_mean_array) / sigma_cos_array

    # standard normal PDF
    phi = (1/np.sqrt(2*np.pi)) * np.exp(-0.5 * y**2)

    # normalization factor: 1 - Φ(0 | μ, σ) = Φ(μ/σ)
    norm_factor = 0.5 * (1 + erf(DM_cos_mean_array / (sigma_cos_array * np.sqrt(2))))

    # truncated Gaussian
    P_cos = (1 / sigma_cos_array) * phi / norm_factor
    P_cos = np.where(DM_cos_array < 0, 0.0, P_cos)

    P_host = (1.0 / (np.sqrt(2 * np.pi) * sigma_host * DM_host_array)) \
             * np.exp(-0.5 * ((np.log(DM_host_array) - mu_host) / sigma_host)**2)

    P_halo_array = P_halo_grid[None, :, None]

    integrand = P_cos * P_host * P_halo_array                     # (N, n_halo, n_host)
    inner     = trapezoid(integrand, x=DM_host_grid, axis=2)      # -> (N, n_halo)
    P_total   = trapezoid(inner, x=DM_halo_grid, axis=1)          # -> (N,)

    return np.maximum(P_total, 1e-100)


# Joint likelihood
def lnlike_cc_sne_bao_frb(theta,
                          z_cc, H_obs, H_err,
                          z_sne, mB_obs, Cinv_sne,
                          z_bao, d_obs, q_bao, Cinv_bao,
                          z_frb, DM_ext_obs):
    
    H0, Om0, Ob, MB, sigma_host, exp_mu = theta
  
    # --- CC ---
    H_model = H_LCDM(z_cc, H0, Om0)
    if not np.all(np.isfinite(H_model)):
        return -np.inf
    lnL_cc =  -0.5 * np.sum(((H_obs - H_model) / H_err)**2)

    # --- SNe ---
    mB_model = mB_LCDM(z_sne, H0, Om0, MB)
    if not np.all(np.isfinite(mB_model)):
        return -np.inf
    resid_sne = mB_obs - mB_model
    lnL_sne   = -0.5 * (resid_sne @ Cinv_sne @ resid_sne)

    # --- BAO ---
    d_model = d_LCDM(z_bao, q_bao, H0, Om0, Ob)
    if not np.all(np.isfinite(d_model)):
        return -np.inf
    resid_bao = d_obs - d_model
    lnL_bao   = -0.5 * (resid_bao @ Cinv_bao @ resid_bao)

    # --- FRB ---
    DM_cos_mean = DM_cos_LCDM(z_frb, H0, Om0, Ob)
    if not np.all(np.isfinite(DM_cos_mean)):
        return -np.inf
    # Statistical (large-scale-structure) scatter, as in the fiducial model
    sigma_cos_stat = 173.8 * (z_frb**0.4)
    # DMbudget robustness test: DM_MW,ISM (NE2001) systematic uncertainty added
    # in quadrature (sigma_MW, defined in the Fiducial/Priors cell).
    # DM_MW,halo uncertainty is instead marginalized explicitly via
    # P_halo(DM_halo) inside P_DM_ext_vectorized (Likelihood-3-style),
    # so it is NOT included here in quadrature.
    sigma_cos = np.sqrt(sigma_cos_stat**2 + sigma_MW**2)
    P_total   = P_DM_ext_vectorized(z_frb, DM_ext_obs, DM_cos_mean,
                                     sigma_cos, sigma_host, exp_mu)
    lnL_frb   = np.sum(np.log(P_total))
    return lnL_cc + lnL_sne + lnL_bao + lnL_frb


# Joint posterior
def lnprob_cc_sne_bao_frb(theta,
                          z_cc, H_obs, H_err,
                          z_sne, mB_obs, Cinv_sne,
                          z_bao, d_obs, q_bao, Cinv_bao,
                          z_frb, DM_ext_obs):
    lp = lnprior_cc_sne_bao_frb(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + lnlike_cc_sne_bao_frb(theta,
                                      z_cc, H_obs, H_err,
                                      z_sne, mB_obs, Cinv_sne,
                                      z_bao, d_obs, q_bao, Cinv_bao,
                                      z_frb, DM_ext_obs)


# ====================================================
# Quick test
# ====================================================

theta_test = [70.0, 0.3, 0.0495, -19.24, 0.7, 120.0]
print("CC + SNe + BAO + FRB log-posterior =",
       lnprob_cc_sne_bao_frb(theta_test,
                             z_cc, H_obs, H_err,
                             z_sne, mB_obs, Cinv_sne,
                             z_bao, d_obs, q_bao, Cinv_bao,
                             z_frb, DM_ext_obs))

CC + SNe + BAO + FRB log-posterior = -1840.3544552373755


In [9]:
# ====================================================
# MCMC configuration
# ====================================================

ndim     = 6
nwalkers = 48
# DMbudget+CCsubset robustness test: nsteps reduced from 40000 -> 6000. The double
# DM_halo x DM_host convolution (DMbudget, see likelihood cell) is much more
# expensive per step than the fiducial single-convolution model, and the
# fiducial run's own diagnostics (tau_max ~ 71, N_eff ~ 26800 at 40000
# steps) show 40000 steps was far more than needed for convergence
# (length_ratio >> 50). At 6000 steps, length_ratio ~ 6000/71 ~ 84,
# still comfortably above the >=50 convergence criterion, assuming a
# similar tau_max for this smoother (fully marginalized) likelihood.
# Dropping the two correlated CC points (CCsubset) has a negligible effect on
# per-step cost (CC likelihood is O(33) vs. O(35), trivial next to the
# FRB double-convolution term), so this budget is unchanged.
nsteps   = 6000

theta0 = np.array([70.0, 0.3, 0.0495, -19.24, 0.7, 120.0])
scales = np.array([ 2.0, 0.02, 0.001,   0.02, 0.1,  10.0])

rng = np.random.default_rng(42)
p0  = theta0 + scales * rng.standard_normal((nwalkers, ndim))

for i in range(nwalkers):
    while not np.isfinite(lnprior_cc_sne_bao_frb(p0[i])):
        p0[i] = theta0 + scales * rng.standard_normal(ndim)


# ====================================================
# MCMC run
# ====================================================

print(f"\n=== Running MCMC (CC + SNe + BAO + FRB) ===")
t_mcmc = time.time()

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, lnprob_cc_sne_bao_frb,
    args=(z_cc, H_obs, H_err, z_sne, mB_obs, Cinv_sne,
          z_bao, d_obs, q_bao, Cinv_bao, z_frb, DM_ext_obs)
)
sampler.run_mcmc(p0, nsteps, progress=True)

print(f"\nMCMC completed in {(time.time() - t_mcmc) / 60:.1f} min")


=== Running MCMC (CC + SNe + BAO + FRB) ===


100%|██████████| 6000/6000 [32:10<00:00,  3.11it/s]


MCMC completed in 32.2 min


In [10]:
# ====================================================
# Convergence diagnostics
# ====================================================

print("\n=== Convergence diagnostics ===")

chain        = sampler.get_chain()
nsteps_total = chain.shape[0]

acc_frac = np.mean(sampler.acceptance_fraction)
print(f"Mean acceptance fraction: {acc_frac:.3f}")
if   0.2  <= acc_frac <= 0.5:  acc_status = "IDEAL"
elif 0.15 <= acc_frac <= 0.6:  acc_status = "ACCEPTABLE"
else:                           acc_status = "PROBLEMATIC"
print(f"Acceptance status: {acc_status}")

tau          = sampler.get_autocorr_time(tol=0)
tau_max      = np.max(tau)
length_ratio = nsteps_total / tau_max
print(f"\nMax autocorrelation time τ_max = {tau_max:.2f}")
print(f"Chain length / τ_max           = {length_ratio:.1f}")
length_status = "PASSED" if length_ratio >= 50 else "FAILED"
print(f"Length criterion (>50×τ)       : {length_status}")

nburn     = int(3 * tau_max)
nthin     = max(1, int(tau_max / 2))
n_post    = nsteps_total - nburn
N_eff     = nwalkers * (n_post / tau)
N_eff_min = np.min(N_eff)
print(f"\nBurn-in (3×τ_max)              = {nburn} steps ({100*nburn/nsteps_total:.1f}%)")
print(f"Thin (τ_max/2)                 = {nthin}")
print(f"Minimum N_eff                  = {int(N_eff_min)}")
if   N_eff_min >= 2000: eff_status = "EXCELLENT"
elif N_eff_min >= 1000: eff_status = "VERY GOOD"
elif N_eff_min >= 500:  eff_status = "ACCEPTABLE"
else:                   eff_status = "LOW"
print(f"Sampling quality               : {eff_status}")

if acc_status != "PROBLEMATIC" and length_status == "PASSED" and N_eff_min >= 500:
    print("\nCONVERGENCE STATUS: ✅ PASSED")
else:
    print("\nCONVERGENCE STATUS: ❌ NOT RELIABLE")


=== Convergence diagnostics ===
Mean acceptance fraction: 0.511
Acceptance status: ACCEPTABLE

Max autocorrelation time τ_max = 66.80
Chain length / τ_max           = 89.8
Length criterion (>50×τ)       : PASSED

Burn-in (3×τ_max)              = 200 steps (3.3%)
Thin (τ_max/2)                 = 33
Minimum N_eff                  = 4167
Sampling quality               : EXCELLENT

CONVERGENCE STATUS: ✅ PASSED


In [ ]:
# ====================================================
# Chain extraction and MAP identification
# ====================================================

# Extract flat chains and log-probability (discarding burn-in)
flat_samples  = sampler.get_chain(discard=nburn, flat=True)
log_prob_flat = sampler.get_log_prob(discard=nburn, flat=True)

# --- Identification of Maximum A Posteriori (MAP) ---
map_idx    = np.argmax(log_prob_flat)
map_params = flat_samples[map_idx]

print("\n=== Maximum A Posteriori (MAP) ===")
param_names = ["H0", "Om0", "Ob", "MB", "sigma_host", "exp_mu"]
for name, val in zip(param_names, map_params):
    print(f"{name:10s}: {val:.4f}")
print(f"{'Log-prob':10s}: {log_prob_flat[map_idx]:.4f}")
# ----------------------------------------------------

# Save flat samples (DMbudget+CCsubset combined robustness test -- distinct filenames
# from the DMbudget-only run, which uses the "_DMbudget" suffix in this same folder)
fname = "flat_samples_LCDM_cc_sne_bao_frb_DMbudget_CCsubset.npy"
np.save(fname, flat_samples)

# Save log-posterior (enables MAP in future sessions)
fname_lp = "log_prob_LCDM_cc_sne_bao_frb_DMbudget_CCsubset.npy"
np.save(fname_lp, log_prob_flat)

print(f"\nFinal independent samples : {len(flat_samples)}")
print(f"Burn-in used                : {nburn} steps")
print(f"Saved to                    : {fname}")
print(f"Log-posterior saved to      : {fname_lp}")

In [ ]:
# ====================================================
# Corner plot (GetDist)
# ====================================================

param_names  = ["H0", "Om0", "Ob", "MB", "sigma_host", "exp_mu"]
param_labels = [r"H_0", r"\Omega_{\rm m}", r"\Omega_{\rm b}",
                r"M_B", r"\sigma_{\rm host}", r"e^\mu"]

samples_gd = MCSamples(
    samples=flat_samples,
    names=param_names,
    labels=param_labels
)

samples_gd.updateSettings({
    "smooth_scale_1D": 0.25,
    "smooth_scale_2D": 0.25,
    "fine_bins":       1024,
    "fine_bins_2D":    1024,
})

g = plots.get_subplot_plotter()
g.settings.axes_fontsize       = 14
g.settings.lab_fontsize        = 16
g.settings.legend_fontsize     = 12
g.settings.linewidth_contour   = 1.5
g.settings.num_plot_contours   = 2
g.settings.axis_marker_lw      = 1.0
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add    = 0.3

g.triangle_plot(samples_gd, filled=True,
                legend_labels=["CC (33 pts) + SNe + BAO + FRB"],
                title_limit=0)

# ---- Fill under curve and add titles from samples ----
fmt_map = {"H0": ".2f", "Om0": ".3f", "Ob": ".4f", "MB": ".3f", "sigma_host": ".3f", "exp_mu": ".1f"}

for i, name in enumerate(param_names):
    samples = flat_samples[:, i]
    median  = np.percentile(samples, 50)
    lower68 = np.percentile(samples, 16)
    upper68 = np.percentile(samples, 84)
    ep68    = upper68 - median
    em68    = median  - lower68

    ax   = g.subplots[i, i]
    line = ax.get_lines()[0]
    line.set_linewidth(1.0)

    # Median line
    ax.axvline(median, color="blue", lw=1.0, ls="--", alpha=0.9)

    # Add title from samples
    fmt   = fmt_map[name]
    title = f"${param_labels[i]} = {median:{fmt}}^{{+{ep68:{fmt}}}}_{{-{em68:{fmt}}}}$"
    ax.set_title(title, fontsize=11, pad=4)

plt.subplots_adjust(top=0.95)
figname = "Corner_LCDM_cc_sne_bao_frb_DMbudget_CCsubset.png"
plt.savefig(figname, dpi=300, bbox_inches="tight")
plt.show()

In [13]:
# ====================================================
# Marginalized statistics
# ====================================================

param_names_indices = [("H0", 0), ("Om0", 1), ("Ob", 2), ("MB", 3), ("sigma_host", 4), ("exp_mu", 5)]
fmt_map = {"H0": ".2f", "Om0": ".3f", "Ob": ".4f", "MB": ".3f", "sigma_host": ".3f", "exp_mu": ".1f"}

print("\n" + "="*60)
print(f"{'Final DMbudget+CCsubset Robustness Test Results (LCDM | CC(33)+SNe+BAO+FRB)':^60}")
print("="*60)

for name, idx in param_names_indices:
    samples = flat_samples[:, idx]
    median  = np.percentile(samples, 50)
    lower68 = np.percentile(samples, 16)
    upper68 = np.percentile(samples, 84)
    lower95 = np.percentile(samples, 2.5)
    upper95 = np.percentile(samples, 97.5)
    ep68    = upper68 - median
    em68    = median  - lower68
    ep95    = upper95 - median
    em95    = median  - lower95
    fmt     = fmt_map[name]
    print(f"\n  {name}:")
    print(f"    {median:{fmt}} +{ep68:{fmt}} -{em68:{fmt}}  (68% CL)")
    print(f"    {median:{fmt}} +{ep95:{fmt}} -{em95:{fmt}}  (95% CL)")

print("\n" + "="*60)


Final DMbudget+CCsubset Robustness Test Results (LCDM | CC(33)+SNe+BAO+FRB)

  H0:
    70.51 +1.05 -1.09  (68% CL)
    70.51 +2.08 -2.19  (95% CL)

  Om0:
    0.305 +0.008 -0.008  (68% CL)
    0.305 +0.016 -0.015  (95% CL)

  Ob:
    0.0502 +0.0017 -0.0017  (68% CL)
    0.0502 +0.0034 -0.0034  (95% CL)

  MB:
    -19.336 +0.032 -0.034  (68% CL)
    -19.336 +0.063 -0.069  (95% CL)

  sigma_host:
    0.799 +0.121 -0.109  (68% CL)
    0.799 +0.243 -0.199  (95% CL)

  exp_mu:
    102.8 +17.0 -16.1  (68% CL)
    102.8 +33.8 -30.4  (95% CL)



In [14]:
# ====================================================
# Walker chains
# ====================================================

chain_plot = sampler.get_chain().copy()

fig, axes = plt.subplots(6, figsize=(10, 8), sharex=True)
for i, label in enumerate(["H0", "Om0", "Ob", "MB", "sigma_host", "exp_mu"]):
    axes[i].plot(chain_plot[:, :, i], alpha=0.3, lw=0.5)
    axes[i].set_ylabel(label)
axes[-1].set_xlabel("Step")

plt.tight_layout()
plt.show()

In [15]:
# ========================================================
# Derived Parameter: Sound Horizon rd
# ========================================================

def compute_rd(H0, Om0, Ob):
    h2   = (H0 / 100.0)**2
    Obh2 = Ob * h2
    Omh2 = Om0 * h2
    rd   = 147.05 * ((Obh2 / 0.02236)**-0.13) \
                  * ((Omh2 / 0.1432)**-0.23)  \
                  * ((Neff  / 3.04)**-0.1)
    return rd

# Compute rd for entire posterior
# flat_samples columns: [H0, Om0, Ob, MB, sigma_host, exp_mu]
rd_chain = np.array([
    compute_rd(s[0], s[1], s[2]) for s in flat_samples
])

mean_rd  = np.mean(rd_chain)
std_rd   = np.std(rd_chain)
lower_95 = np.percentile(rd_chain, 2.5)
upper_95 = np.percentile(rd_chain, 97.5)

print("\n" + "="*60)
print(f"{'DERIVED SOUND HORIZON':^60}")
print("="*60)
print(f"  r_d = {mean_rd:.3f} ± {std_rd:.3f} Mpc (68% CL)")
print(f"  95% CL : [{lower_95:.3f}, {upper_95:.3f}] Mpc")
print("="*60)


                   DERIVED SOUND HORIZON                    
  r_d = 143.093 ± 2.210 Mpc (68% CL)
  95% CL : [138.900, 147.608] Mpc


In [ ]:
# ====================================================
# Model selection statistics
# ====================================================

# Toggle: load samples from file or use sampler from memory
LOAD_FROM_FILE = False   # True = load from .npy | False = use sampler in memory

if LOAD_FROM_FILE:
    fname        = "flat_samples_LCDM_cc_sne_bao_frb_DMbudget_CCsubset.npy"
    fname_lp     = "log_prob_LCDM_cc_sne_bao_frb_DMbudget_CCsubset.npy"
    flat_samples  = np.load(fname)
    log_prob_flat = np.load(fname_lp)
    idx_map  = np.argmax(log_prob_flat)
    theta_bf = flat_samples[idx_map]
    print(f"Loaded {len(flat_samples)} samples from {fname}")
    print(f"Best-fit (MAP): {theta_bf}")

else:
    idx_map  = np.argmax(sampler.get_log_prob(discard=nburn, flat=True))
    theta_bf = flat_samples[idx_map]
    print(f"Best-fit (MAP): {theta_bf}")

theta_bf_like = theta_bf.copy()

# Maximum log-likelihood
lnL_max = lnlike_cc_sne_bao_frb(
    theta_bf_like,
    z_cc, H_obs, H_err,
    z_sne, mB_obs, Cinv_sne,
    z_bao, d_obs, q_bao, Cinv_bao,
    z_frb, DM_ext_obs
)

# Number of data points
N_cc  = len(z_cc)
N_sne = len(z_sne)
N_bao = len(z_bao)
N_frb = len(z_frb)
N_tot = N_cc + N_sne + N_bao + N_frb

# Number of free parameters
k = flat_samples.shape[1]

# Chi-squared
chi2     = -2.0 * lnL_max
dof      = N_tot - k
chi2_red = chi2 / dof

# AIC and BIC
AIC = 2.0 * k - 2.0 * lnL_max
BIC = k * np.log(N_tot) - 2.0 * lnL_max

print("\n" + "="*60)
print(f"{'MODEL SELECTION STATISTICS':^60}")
print(f"{'(LCDM: CC(33, CCsubset-corrected) + SNe + BAO + FRB)':^60}")
print("="*60)
print(f"\n  Data points:")
print(f"    CC       : {N_cc}")
print(f"    SNe      : {N_sne}")
print(f"    BAO      : {N_bao}")
print(f"    FRB      : {N_frb}")
print(f"    Total    : {N_tot}")
print(f"\n  Free parameters (k) : {k}")
print(f"  Degrees of freedom  : {dof}")
print(f"\n  ln L_max  = {lnL_max:.4f}")
print(f"  χ²        = {chi2:.4f}")
print(f"  χ²/dof    = {chi2_red:.4f}")
print(f"  AIC       = {AIC:.4f}")
print(f"  BIC       = {BIC:.4f}")
print("\n" + "="*60)